In [1]:
"""
DATA MINING - TUGAS BESAR 04
Metode 1: REGRESI untuk memprediksi Sales Volume (dataset Zara)

Alur:
  1. Load data
  2. Data cleaning & pemilihan fitur
  3. Split train/test
  4. Pipeline preprocessing + 3 model regresi
  5. Evaluasi (MAE, RMSE, R2, Cross-Validation)
  6. Skenario B: model TANPA 'Promotion' (cek ketergantungan satu fitur)
  7. Feature importance (permutation importance)
  8. Simpan grafik & tabel hasil
"""

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # hapus baris ini jika ingin plt.show() di Jupyter/PC
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# KONFIGURASI  (ubah path sesuai lokasi file Anda)
# ----------------------------------------------------------------------
DATA_PATH = "Business_sales_EDA.csv"
OUTPUT_DIR = "Business_sales_EDAhasil.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "Sales Volume"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)              # jika CSV: pd.read_csv("file.csv")
print("Ukuran data :", df.shape)
print("Missing value pada kolom fitur akan ditangani otomatis oleh pipeline.\n")

# ----------------------------------------------------------------------
# 2. PEMILIHAN FITUR
#    Dibuang: identifier/teks (Product ID, url, name, description)
#             kolom konstan (Product Category, brand, currency)
# ----------------------------------------------------------------------
NUM_FEATURES = ["price"]
CAT_FEATURES = [
    "Product Position", "Promotion", "Seasonal",
    "terms", "section", "season", "material", "origin",
]
FEATURES = NUM_FEATURES + CAT_FEATURES

data = df[FEATURES + [TARGET]].dropna().drop_duplicates()
X = data[FEATURES]
y = data[TARGET]
print(f"Data dipakai : {len(data)} baris, {len(FEATURES)} fitur")
print(f"Target       : {TARGET} (min={y.min()}, max={y.max()}, mean={y.mean():.1f})\n")

# ----------------------------------------------------------------------
# 3. SPLIT TRAIN / TEST
# ----------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} baris | Test: {X_test.shape[0]} baris\n")


# ----------------------------------------------------------------------
# 4. PREPROCESSING + MODEL
# ----------------------------------------------------------------------
def build_preprocessor(num_cols, cat_cols):
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )


def get_models():
    return {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(
            n_estimators=100, max_depth=12, min_samples_leaf=5,
            n_jobs=-1, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=100, learning_rate=0.1, max_depth=3,
            random_state=RANDOM_STATE),
    }


def evaluate_models(X_tr, X_te, y_tr, y_te, num_cols, cat_cols, label):
    """Latih semua model, kembalikan tabel metrik + pipeline terlatih."""
    rows, fitted = [], {}
    kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    for name, model in get_models().items():
        pipe = Pipeline([
            ("prep", build_preprocessor(num_cols, cat_cols)),
            ("model", model),
        ])
        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_te)

        cv_r2 = cross_val_score(pipe, X_tr, y_tr, cv=kf, scoring="r2")

        rows.append({
            "Skenario": label,
            "Model": name,
            "MAE": mean_absolute_error(y_te, pred),
            "RMSE": np.sqrt(mean_squared_error(y_te, pred)),
            "R2 (test)": r2_score(y_te, pred),
            "R2 CV mean": cv_r2.mean(),
            "R2 CV std": cv_r2.std(),
        })
        fitted[name] = pipe
        print(f"[{label}] {name:<18} R2={rows[-1]['R2 (test)']:.4f} "
              f"MAE={rows[-1]['MAE']:.2f} RMSE={rows[-1]['RMSE']:.2f}")
    return pd.DataFrame(rows), fitted


# ----------------------------------------------------------------------
# 5. SKENARIO A: SEMUA FITUR
# ----------------------------------------------------------------------
print("=" * 70)
print("SKENARIO A - Semua fitur")
print("=" * 70)
res_a, models_a = evaluate_models(
    X_train, X_test, y_train, y_test, NUM_FEATURES, CAT_FEATURES, "A: Semua fitur")

# ----------------------------------------------------------------------
# 6. SKENARIO B: TANPA 'Promotion'
#    Tujuan: melihat apakah fitur lain (harga, musim, dll.) tetap berguna.
# ----------------------------------------------------------------------
print("\n" + "=" * 70)
print("SKENARIO B - Tanpa fitur 'Promotion'")
print("=" * 70)
cat_b = [c for c in CAT_FEATURES if c != "Promotion"]
res_b, _ = evaluate_models(
    X_train.drop(columns="Promotion"), X_test.drop(columns="Promotion"),
    y_train, y_test, NUM_FEATURES, cat_b, "B: Tanpa Promotion")

# ----------------------------------------------------------------------
# 7. TABEL PERBANDINGAN
# ----------------------------------------------------------------------
results = pd.concat([res_a, res_b], ignore_index=True).round(4)
results.to_csv(f"{OUTPUT_DIR}/perbandingan_model.csv", index=False)
print("\n" + "=" * 70)
print("PERBANDINGAN SEMUA MODEL")
print("=" * 70)
print(results.to_string(index=False))

best_name = res_a.sort_values("R2 (test)", ascending=False).iloc[0]["Model"]
best_pipe = models_a[best_name]
print(f"\nModel terbaik (Skenario A, berdasarkan R2 test): {best_name}")

# ----------------------------------------------------------------------
# 8. FEATURE IMPORTANCE (permutation, per fitur asli)
# ----------------------------------------------------------------------
perm = permutation_importance(
    best_pipe, X_test, y_test, n_repeats=5,
    random_state=RANDOM_STATE, scoring="r2")
imp = (pd.DataFrame({"Fitur": FEATURES,
                     "Importance": perm.importances_mean,
                     "Std": perm.importances_std})
       .sort_values("Importance", ascending=False))
imp.round(4).to_csv(f"{OUTPUT_DIR}/feature_importance.csv", index=False)
print("\nPERMUTATION IMPORTANCE (penurunan R2 jika fitur diacak):")
print(imp.round(4).to_string(index=False))

# ----------------------------------------------------------------------
# 9. VISUALISASI
# ----------------------------------------------------------------------
pred_best = best_pipe.predict(X_test)
resid = y_test - pred_best

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# (a) Actual vs Predicted
ax[0].scatter(y_test, pred_best, alpha=0.3, s=10)
lim = [y.min(), y.max()]
ax[0].plot(lim, lim, "r--")
ax[0].set(title=f"Actual vs Predicted ({best_name})",
          xlabel="Actual Sales Volume", ylabel="Predicted Sales Volume")

# (b) Distribusi residual
ax[1].hist(resid, bins=40, edgecolor="black")
ax[1].axvline(0, color="r", ls="--")
ax[1].set(title="Distribusi Residual", xlabel="Residual", ylabel="Frekuensi")

# (c) Feature importance
imp_plot = imp.sort_values("Importance")
ax[2].barh(imp_plot["Fitur"], imp_plot["Importance"], xerr=imp_plot["Std"])
ax[2].set(title="Permutation Importance", xlabel="Penurunan R2")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/evaluasi_regresi.png", dpi=150)
plt.close()

# Perbandingan R2 antar skenario
fig, ax = plt.subplots(figsize=(9, 5))
pivot = results.pivot(index="Model", columns="Skenario", values="R2 (test)")
pivot.plot(kind="bar", ax=ax)
ax.set(title="R2 (test) per Model dan Skenario", ylabel="R2")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/perbandingan_skenario.png", dpi=150)
plt.close()

# ----------------------------------------------------------------------
# 10. CONTOH PREDIKSI DATA BARU
# ----------------------------------------------------------------------
contoh = pd.DataFrame([{
    "price": 49.9, "Product Position": "End-cap", "Promotion": "Yes",
    "Seasonal": "Yes", "terms": "jackets", "section": "WOMAN",
    "season": "Winter", "material": "Wool", "origin": "Turkey",
}])
print(f"\nContoh prediksi Sales Volume: {best_pipe.predict(contoh)[0]:.0f} unit")

print(f"\nSelesai. Hasil disimpan di: {OUTPUT_DIR}")

Ukuran data : (20252, 17)
Missing value pada kolom fitur akan ditangani otomatis oleh pipeline.

Data dipakai : 20249 baris, 9 fitur
Target       : Sales Volume (min=518, max=1940, mean=1097.4)

Train: 16199 baris | Test: 4050 baris

SKENARIO A - Semua fitur
[A: Semua fitur] Linear Regression  R2=0.9269 MAE=62.70 RMSE=79.65
[A: Semua fitur] Ridge Regression   R2=0.9269 MAE=62.70 RMSE=79.65
[A: Semua fitur] Random Forest      R2=0.9313 MAE=60.61 RMSE=77.18
[A: Semua fitur] Gradient Boosting  R2=0.9340 MAE=59.42 RMSE=75.65

SKENARIO B - Tanpa fitur 'Promotion'
[B: Tanpa Promotion] Linear Regression  R2=0.2501 MAE=228.30 RMSE=255.04
[B: Tanpa Promotion] Ridge Regression   R2=0.2501 MAE=228.30 RMSE=255.04
[B: Tanpa Promotion] Random Forest      R2=0.5564 MAE=147.36 RMSE=196.15
[B: Tanpa Promotion] Gradient Boosting  R2=0.4413 MAE=186.18 RMSE=220.14

PERBANDINGAN SEMUA MODEL
          Skenario             Model      MAE     RMSE  R2 (test)  R2 CV mean  R2 CV std
    A: Semua fitur Linear Re

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b470653c-dfdd-474f-9920-933c569b11b8' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>